# Initialize

First, let's import the necessary dependencies and initialize some objects required for the reconstruction.

- The Archive object contains all the raw data and necessary information to perform an image reconstruction. Some objects will require parameters from the archive to initialize.
- The Dicom object allows us to save any reconstructed images as dicom images.
- The ChannelCombiner, Transformer, Gradwarp, Orientation, Calibration and Pure objects will all be used to perform the reconstruction.

In [ ]:
%matplotlib inline

from GERecon import Archive, ChannelCombiner, Dicom, Gradwarp, Orientation, Transformer, Calibration, Pure

import os
import shutil
import numpy as np
import matplotlib.pyplot as plt

archive_filename = os.path.join(os.path.abspath(''), "Data/Pure/ScanArchive.h5")
dicom_out_path = os.path.join(os.path.abspath(''), "Data/Pure/DICOM_OUT")

# Pure-IDXXXXX.h5 file appears in __file__/<exam number> when the archive is loaded. This is the calibration
# file. In this case, that file has just been renamed and moved somewhere more convenient.
pure_calibration_filename = os.path.join(os.path.abspath(''), "Data/Pure/PureCalibration_ScanArchive.h5")

# Clear the output directory
if os.path.isdir(dicom_out_path):
    shutil.rmtree(dicom_out_path, ignore_errors=True)
    
archive = Archive(archive_filename)
dicom = Dicom(archive)
channel_combiner = ChannelCombiner(archive.ChannelCombinerParams())
transform = Transformer(archive.TransformerParams())
gradwarp = Gradwarp()
orient = Orientation()
pure = Pure(archive.PureParams(), pure_calibration_filename)

Next, let's read some important data from the archive to use during the reconstruction.

- x_res, y_res: The x and y resolution of the acquired data; the resolution of kSpace.
- num_control: The number of control packets.
- num_channels: The number of channels used to acquire the data.
- num_passes: The number of passes in the acquisition.
- slices_per_pass: How many slices of data are in each pass.

In [ ]:
metadata = archive.Metadata()
x_res = metadata["acquiredXRes"]
y_res = metadata["acquiredYRes"]
num_control = metadata["controlCount"]
num_channels = metadata["numChannels"]
num_passes = metadata["passes"]
slices_per_pass = archive.SlicesPerPass()

We need to track some data throughout the reconstruction process, including the current pass number and how many slices are in each pass. We also need to track kSpace since kSpace data is not filled in all at once, but with multiple data frames. Note that declaring the data type is important here.

In [ ]:
pass_num = 0
num_slices = slices_per_pass[pass_num]
kspace = np.zeros([x_res, y_res, num_channels, num_slices], dtype=np.complex64)

# Reconstruct

Now we're ready to start reconstructing images. For each control packet, we'll check whether it's a programmable control packet or a scan control packet. If it's programmable, that means there's a data frame containing acquired data that we need to extract. If it's a scan control packet, that means we have all the data required to perform a reconstruction.

In [ ]:
# Loop over all the control packets
for i in range(num_control):

    # Retrieve the next control packet 
    control = archive.NextControl()
    
    # This is a raw control packet, get the next frame so that the control and the frames are in sync.
    # But do not use this frame to fill kspace.
    if control["opcode"] == 16:
        next_frame = np.squeeze(archive.NextFrame())

    # This is a programmable control packet, so use next frame to fill a line of kspace.
    elif control["opcode"] == 1 and 0 < control["viewNum"] <= y_res and control["sliceNum"] < num_slices:
        # Frame data is organized as: ReadoutSize x NumChannels x NumFrames
        # where NumFrames is the number of frames corresponding to this control
        # packet. Each programmable packet corresponds to a single frame. Thus,
        # for this example, the frames dimension will always have a size of 1
        next_frame = np.squeeze(archive.NextFrame())

        # Place the frame into kSpace
        kspace[:, control["viewNum"] - 1, :, control["sliceNum"]] = next_frame

    # This is a scan control packet, so reconstruct the acquired data
    elif control["opcode"] == 0:
        
        # Reconstruct each slice
        for slice_num in range(num_slices):
            
            # Transform each slice of kSpace: this outputs an array organized as ImageXRes x ImageYRes x NumChannels
            # where each inidividual slice has been transformed into image space
            transformed_images = transform.Execute(kspace[:, :, :, slice_num])

            # Combine all the images of the same slice but different channels together into one image, so this
            # outputs an array orgnaizd as ImageXRes x ImageYRes
            combined_image = channel_combiner.SumOfSquares(transformed_images)

            # Create magnitude image
            magnitude_image = np.absolute(combined_image)

            # Get corners for this slice location
            corners = archive.Corners(slice_num, pass_num)

            # Get orientation for this slice location
            orientation = archive.Orientation(slice_num, pass_num)
            
            # Apply Pure Processing
            pured_image = pure.Apply(magnitude_image, corners)

            # Apply gradwarp
            gradwarped_image = gradwarp.Execute(pured_image, corners, fovScalingParams=archive.GradwarpParams())

            # Orient the image
            oriented_image = orient.OrientImage(gradwarped_image, orientation)

            # Display oriented image
            plt.figure()
            plt.imshow(oriented_image, cmap="gray")
            plt.suptitle("Oriented Image: Slice[{}] Pass[{}]".format(slice_num + 1, pass_num + 1))

            # Write the image to dicom
            dicom.Write("{}/image_{}_{}.dcm".format(dicom_out_path, slice_num + 1, pass_num + 1), oriented_image, slice_num,
                        corners, orientation)

        # Receiving a scan control packet means that an acquisition finished,
        # so increase pass number and reset kspace to prepare for next acquisition (if it exists)
        pass_num += 1
        if pass_num < num_passes:
            num_slices = slices_per_pass[pass_num]
            kspace = np.zeros([x_res, y_res, num_channels, num_slices], dtype=np.complex64)

plt.show()